In [2]:
import numpy as np
import os
import xarray as xr
import glob
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import BoundaryNorm
import matplotlib.colors as mcolors
import dask.array as da
import pickle
from scipy.stats import t
import matplotlib.ticker as ticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.cm as cm

In [3]:
# removes underground values, contains advections and mean wind profiles

with open('/home/annierosen16/master_clean.pkl', 'rb') as f:
    
    master = pickle.load(f)

In [4]:
mean_surface_pressure = np.nanmean(master['average_sp']) / 100

931.9282427656697

In [5]:
# reading in ERA5 u, v, w and q

base_path = '/data/rong4/Data/ERA5/3hourly/quvw_US'

years = [str(year) for year in range(2001, 2019)]

def get_files(folder, component):

    files = glob.glob(os.path.join(base_path, folder, f"era5.{component}.*.nc"))

    filtered_files = [f for f in files if any(year in f for year in years)]
    
    return filtered_files

# Get the files for each component

q_files = get_files('specific_humidity', 'specific_humidity')

# open all datasets at once

q = xr.open_mfdataset(q_files, combine='by_coords', chunks={'time': 24})

In [7]:
q_sgp = q.sel(
    
    latitude=slice(39, 30),
    
    longitude=slice(-105, -95)
)

q_sgp['time'] = q_sgp['time'] - pd.Timedelta(hours=6)

q_sgp = q_sgp.where(
    
    (q_sgp.time.dt.month.isin([5, 6, 7, 8, 9])) &
    
    (q_sgp.time.dt.year.isin(range(2001, 2019))),
    
    drop=True
)

In [8]:
master.columns

Index(['date', 'latitude', 'longitude', 'APE', 'daily_precip',
       'reason_for_fail', 'total_rainfall', 'saturation', 'ctp', 'hilow',
       'blc', 'blt', 'day_month', 'ctp_mean', 'ctp_std', 'ctp_count',
       'hilow_mean', 'hilow_std', 'hilow_count', 'ctp_stnd_anom',
       'hilow_stnd_anom', 'blc_mean', 'blc_std', 'blc_count', 'blt_mean',
       'blt_std', 'blt_count', 'blt_stnd_anom', 'blc_stnd_anom', 'dryape',
       'wetape', 'wetcoupling', 'drycoupling', 'zonal_advection',
       'meridional_advection', 'vertical_advection', 'average_sp'],
      dtype='object')

In [10]:
dryapes = master[master['dryape']==True]

wetapes = master[master['wetape']==True]

dryapes = dryapes[['latitude','longitude','date']]

wetapes = wetapes[['latitude','longitude','date']]

In [15]:
q_sgp['date'] = q_sgp['time'].date()

AttributeError: 'DataArray' object has no attribute 'date'